In [205]:
import pandas as pd
from openai import OpenAI
import os
import json
import numpy as np

In [ ]:
APIKEY=""

In [207]:
client = OpenAI(api_key=APIKEY)


In [208]:
USER="9280"

In [209]:
user_path=rf"C:\Users\Franco\Desktop\Pubblicazione\04\utenti_data\{USER}_data.csv"
segment_user=rf"C:\Users\Franco\Desktop\Pubblicazione\05.1\segment\user_breaks\{USER}_breaks.csv"


In [210]:
df = pd.read_csv(user_path)  


In [211]:


df["mean_score"] = 2*df["Prob_Severe_Depressed"]+df["Prob_Moderate_Depressed"]

df["Date"] = pd.to_datetime(df["Date"]).dt.date
mean_score = df.groupby("Date")["mean_score"].mean().sort_index()



In [212]:
breaks= pd.read_csv(segment_user)



In [213]:
posts=df['Text']
timestamps=df['Date']



In [214]:
def build_prompt_base(posts, timestamps):
    prompt = f"""
Posts ($p$): {posts}
Timestamps ($t$): {timestamps}

Task:

You are given a series of social media posts $p$ written by a single user, sorted by timestamp $t$.
Each post in $p$ is associated with a timestamp $t$.
Analyze the posts $p$ as a time series and describe the user’s emotional state across the entire period.
Identify any posts that show signs of depressive-leaning emotional expressions and explain why those posts stand out in context.
Describe the period in which these emotions appear most strongly and how the emotional tone changes from beginning to end.
Use the content of the posts $p$ to create a clear, coherent, and integrative narrative about the evolution of the user’s emotional state over time.

Output:

The output must be limited exclusively to a single integrative analytical narrative describing the user’s emotional evolution.
No bullet points, lists, section headings, or meta-commentary should be included.

"""
    return prompt.strip()

In [215]:

def build_prompt_trajectory(posts, timestamps, mean_scores,phase, delta):
    prompt = f"""
Inputs:

Posts ($p$): {posts}
Timestamps ($t$): {timestamps}
Daily mean scores ($c$): {mean_scores}
Phase number ($n$): {phase}
Delta value ($d$): {delta}

Task:

You are given a series of social media posts $p$ written by a single user, sorted by timestamp $t$.
Each post in $p$ is associated with a timestamp $t$. The series $c$ contains the daily mean scores for each day represented in $t$.
All posts belong exclusively to Phase $n$ of the user’s timeline.

Analyze the posts $p$ as a time series and describe the user’s emotional state across the entire period represented by Phase $n$.

Base the analysis on the explicit semantic, emotional, and expressive content of the posts.
Use the daily mean scores $c$ and the delta value $d$ strictly as auxiliary contextual signals to support interpretation of emotional intensity and directional tendency within Phase $n$. Do not treat them as labels, diagnoses, or primary decision criteria.

Identify any posts that exhibit depressive-leaning emotional expressions and explain why they are salient in comparison to other posts within the same phase.

Describe when these emotional expressions are most pronounced and explain how emotional tone, intensity, or outlook changes from the beginning to the end of Phase $n$, incorporating the general directional information conveyed by delta $d$ while allowing for gradual, uneven, or internally mixed emotional progression.

Synthesize the content of the posts into a single, coherent analytical narrative that reconstructs the emotional evolution contained entirely within Phase $n$, without extrapolation beyond the provided timestamps.

Output format (must be followed exactly):

Phase $n$ (from <computed_start_date> to <computed_end_date>):
 <one analytically coherent paragraph describing this phase>

Constraints:

Dates MUST be computed strictly from timestamps $t$.
Produce exactly one paragraph.
The paragraph MUST be no longer than 100 words.
Do NOT reference content, phases, or emotional states outside Phase $n$.
Do NOT introduce diagnostic or clinical conclusions.
Do NOT use bullet points, lists, or headings other than the exact label specified above.
Do NOT include meta-commentary, justifications, or references to the task.
"""
    return prompt.strip()


In [216]:
def build_prompt_trajectory_summary(list_of_phase):
    prompt = f"""
Inputs:

Phase analyses ($P$): {list_of_phase}

Task:

You are given a chronologically ordered list of phase-level analytical narratives $P$, each describing the user’s emotional state during a specific phase of their timeline.

Analyze these phase summaries as a higher-level temporal sequence and produce an integrated summary of the user’s overall emotional trajectory across all phases.

Base the summary on the continuities, shifts, and inflection points described in the phase narratives, explaining how emotional tone, intensity, stability, and outlook evolve from earlier to later phases.

Identify recurring emotional patterns and highlight any phases that represent meaningful changes in direction, escalation, attenuation, or stabilization of emotional distress, as described in the inputs.

Synthesize the information into a single coherent narrative that reflects the longitudinal emotional trajectory, without introducing new interpretations, diagnoses, or assumptions beyond what is supported by the provided phase analyses.

Output:

Produce exactly one integrative analytical paragraph summarizing the user’s emotional trajectory across all phases.

Constraints:
The paragraph MUST be no longer than 200 words.
Do NOT restate individual phase labels verbatim.
Do NOT introduce timestamps or dates not already implied by the phase ordering.
Do NOT use bullet points, lists, or section headings.
Do NOT include meta-commentary or references to the task.
Do NOT introduce clinical or diagnostic conclusions.
        

"""
    return prompt.strip()

In [217]:
def ask_chatgpt(prompt):
    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


In [227]:
prompt_base = build_prompt_base(posts, timestamps)
output_base=ask_chatgpt(prompt_base)

In [230]:
output_base

'Across the span of these posts, what emerges is the story of someone whose online voice begins in a relatively outward-looking, socially engaged way, then gradually shows more frequent flashes of frustration, disillusionment, and low mood, while still punctuated by humor, ordinary interests, and moments of connection. Because the content is varied and often casual, the emotional trajectory is not a simple slide into depression, but rather a pattern of mixed emotional states in which depressive-leaning expressions become more noticeable and, at certain periods, more concentrated.\n\nEarly in the timeline, around 2014, the user’s posts tend to focus on external events and everyday life. Statements about hoping for something “like many of you I’m assuming,” or recounting experiences such as working at Wegmans and meeting notable people, place the user in a social environment where they are comparing themselves with others but still assuming some shared experience and camaraderie. This as

In [ ]:
def ask_phase(posts, timestamps, mean_score, breaks):
   
    timestamps = pd.to_datetime(pd.Series(timestamps))
    posts = pd.Series(posts)
    
    phases = []

    for i in range(len(breaks) - 1):
        start_date = pd.to_datetime(breaks['Date'][i])
        end_date = pd.to_datetime(breaks['Date'][i+1])
        
        mask = (timestamps >= start_date) & (timestamps <= end_date)
        posts_in_phase = posts[mask]
        timestamps_in_phase = timestamps[mask]
        
        delta = float(breaks['score_smooth'][i+1] - breaks['score_smooth'][i])
        prompt_trajectory = build_prompt_trajectory(posts_in_phase.tolist(), 
                                                    timestamps_in_phase.tolist(),
                                                    mean_score, i+1, delta)
        phases.append(ask_chatgpt(prompt_trajectory))
    
    return phases

In [ ]:
list_phase=ask_phase(posts,timestamps,mean_score,breaks)

In [ ]:
prompt_summary = build_prompt_trajectory_summary(list_phase)
output_trajectory_summary=ask_chatgpt(prompt_summary)

In [ ]:
output_trajectory_summary='Overall trajectory:\n'+output_trajectory_summary

In [ ]:
output_trajectory_summary

'Overall trajectory:\nAcross the full sequence, the user moves from generally energetic, socially engaged, and irritation-based posts into a prolonged period of escalating distress dominated by despair and suicidality. Early on, frustrations are situational and quickly redirected into problem-solving and enthusiasm for gadgets and plans. This gradually shifts as withdrawal experiences and later a breakup introduce more sustained sadness, anger, and fear, with emotional intensity no longer confined to specific problems. Over subsequent phases, financial hardship, isolation, and repeated statements about wanting to die become central, with occasional relief from small acts of support or technical interests that never fully offset the prevailing hopelessness. Later phases show the user describing years of unrelenting suffering, failed treatments, anhedonia, and feeling “out of options,” with vivid contemplation of self-destruction and a sense of entrapment. Brief spikes of hope around new

In [ ]:
list_phase.append(output_trajectory_summary)

In [ ]:

base_path = r"C:\Users\Franco\Desktop\Pubblicazione\05.1\prompt\output prompt"  


filename = os.path.basename(user_path)  


user_name = filename.split("_")[0]


analysis_file = os.path.join(base_path, f"{user_name}_base.txt")

with open(analysis_file, "w", encoding="utf-8") as f:
            f.write(output_base)





Cartelle create e analisi salvata correttamente in: C:\Users\Franco\Desktop\Pubblicazione\05.1\prompt\output prompt\9280_basesss.txt


In [ ]:

base_path = r"C:\Users\Franco\Desktop\Pubblicazione\05.1\prompt\output prompt"  # <-- modifica questo


filename = os.path.basename(user_path)  

user_name = filename.split("_")[0]

analysis_file = os.path.join(base_path, f"{user_name}_trajectory.txt")
with open(analysis_file, "w", encoding="utf-8") as f:
    for item in list_phase:
        f.write(item + "\n\n") 



Cartelle create e analisi salvata correttamente in: C:\Users\Franco\Desktop\Pubblicazione\05.1\prompt\output prompt\9280_trajectory.txt
